<a href="https://colab.research.google.com/github/ronakraichandani/MLOAssignment_2026EEY7568_Ronak/blob/main/notebooks/colab_run2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tree-type network of small neural networks - Colab runner
**Runtime -> Change runtime type -> T4 GPU** (or better) before running.
Order: setup -> unit tests -> quick trial (~10-15 min, checks everything works) -> full runs -> report.
Results are copied to Google Drive after each stage so a disconnect does not lose them.

In [ ]:
import torch, subprocess
print(torch.__version__, 'cuda:', torch.cuda.is_available())
print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)

2.11.0+cu130 cuda: True
GPU 0: Tesla T4 (UUID: GPU-7ff9c782-d6e4-d237-988a-4d7b930fd0d1)



## 1. Get the code
Either upload `tree_nn_project.zip` (simplest) or clone your private repo with a personal access token.

In [ ]:
from google.colab import files
up = files.upload()            # choose tree_nn_project.zip
!unzip -q -o tree_nn_project.zip -d /content
%cd /content/tree_nn_project

Saving tree_nn_project.zip to tree_nn_project.zip
/content/tree_nn_project


In [ ]:
# ALTERNATIVE: private GitHub repo (token needs 'repo' scope; do not share this notebook with the token in it)
# TOKEN='ghp_...'; USER='<github-user>'; REPO='<repo>'
# !git clone https://{TOKEN}@github.com/{USER}/{REPO}.git /content/tree_nn_project
# %cd /content/tree_nn_project

In [ ]:
!pip -q install reportlab scikit-learn matplotlib pillow
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/tree_nn_results'
!mkdir -p {SAVE}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.9 MB/s eta 0:00:00
Mounted at /content/drive


## 2. Unit tests of the tree logic (CPU, ~1 min)

In [ ]:
!python tests/test_tree_logic.py

[paper example] nodes=1 acc=1.0
[multi use_B=False] nodes=538 depth=10 params=8107 train acc by depth=[0.814, 0.814, 0.814, 0.814, 0.814, 0.814, 0.814, 0.814, 0.855, 0.945, 1.0]
   pruned(min_pos=5): params=4642 acc=0.874
[multi use_B=True] nodes=871 depth=10 params=13102 train acc by depth=[0.814, 0.814, 0.814, 0.814, 0.814, 0.814, 0.814, 0.857, 0.929, 0.979, 1.0]
   pruned(min_pos=5): params=7477 acc=0.870
[binary] nodes=492 depth=10 params=6405 acc=1.0
ALL TESTS PASSED


## 3. Quick trial - run this BEFORE the long jobs
Small subset, few widths, shallow tree. If anything crashes here, copy the traceback back to Claude.

In [ ]:
!pip -q install datasets
import numpy as np, os
from datasets import load_dataset
os.makedirs('data', exist_ok=True)
ds = load_dataset("uoft-cs/cifar10")
def conv(split):
    return np.stack([np.array(im) for im in ds[split]["img"]]), np.array(ds[split]["label"])
Xtr, ytr = conv("train"); Xte, yte = conv("test")
np.savez("data/cifar10.npz", Xtr=Xtr, ytr=ytr, Xte=Xte, yte=yte)

p = 'tree_nn/data.py'; s = open(p).read()
s = s.replace('def load_cifar10(root):\n    import torchvision\n',
 'def load_cifar10(root):\n    import os\n    c = os.path.join(root, "cifar10.npz")\n    if os.path.exists(c):\n        d = np.load(c)\n        return d["Xtr"], d["ytr"], d["Xte"], d["yte"], CIFAR_CLASSES\n    import torchvision\n')
open(p, 'w').write(s)

README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  120MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 23.9MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

3588

In [ ]:
!python scripts/run_tree.py --dataset cifar10 --n_train 5000 --trunk scratch_cnn --trunk_epochs 2 \
    --widths 0,16 --steps 500 --max_depth 4 --max_nodes 300 --out results/trial_cifar10
!python scripts/minimize.py --run results/trial_cifar10
!python scripts/visualize_rf.py --run results/trial_cifar10 --n_children 4 --max_depth 2

device: cuda
train 5000  test 10000  classes 10  noisy labels 0
  trunk epoch 1/2  loss 1.909  train acc 0.275
  trunk epoch 2/2  loss 1.587  train acc 0.408
features (5000, 512)  (6s)
[node 0 root d=0] n=5000 width=16 errors=1595
  [node 1 A0 d=1] n=4619 width=16 errors=102
  [node 4 A1 d=1] n=4611 width=16 errors=143
  [node 6 A2 d=1] n=4760 width=16 errors=225
  [node 10 A3 d=1] n=4759 width=16 errors=284
  [node 12 A4 d=1] n=4702 width=16 errors=155
  [node 16 A5 d=1] n=4669 width=16 errors=129
  [node 20 A6 d=1] n=4609 width=16 errors=107
  [node 22 A7 d=1] n=4629 width=16 errors=104
  [node 24 A8 d=1] n=4649 width=16 errors=128
  [node 27 A9 d=1] n=4588 width=16 errors=78
{'depth': 0, 'train_acc': 0.681, 'test_acc': 0.5038, 'params': 8398}
{'depth': 1, 'train_acc': 0.681, 'test_acc': 0.5038, 'params': 90668}
{'depth': 2, 'train_acc': 0.81, 'test_acc': 0.4805, 'params': 172936}
{'depth': 3, 'train_acc': 1.0, 'test_acc': 0.4472, 'params': 223825}
DONE  train 1.0000  test 0.4472  no

## 4. Main runs: interpolation (clean labels and 15% label noise)
`--max_nodes` caps the tree size; if the cap is hit the tree may stop short of 100% - check `tree_train_acc` and say so in your report.
Adjust `--trunk_epochs` (more epochs = stronger trunk = smaller tree) and `--widths` as you like.

In [ ]:
!rm -rf results/trial_cifar10
!python scripts/run_tree.py --dataset cifar10 --trunk scratch_cnn --trunk_epochs 3 --out results/cifar10_clean
!python scripts/run_tree.py --dataset cifar10 --trunk scratch_cnn --trunk_epochs 3 --noise 0.15 --out results/cifar10_noise15
!cp -r results {SAVE}/

device: cuda
train 50000  test 10000  classes 10  noisy labels 0
  trunk epoch 1/3  loss 1.406  train acc 0.477
  trunk epoch 2/3  loss 1.049  train acc 0.620
  trunk epoch 3/3  loss 0.901  train acc 0.677
features (50000, 512)  (24s)
[node 0 root d=0] n=50000 width=256 errors=994
  [node 1 A0 d=1] n=45004 width=16 errors=0
  [node 2 A2 d=1] n=45037 width=64 errors=0
  [node 3 A3 d=1] n=45103 width=64 errors=0
  [node 4 A4 d=1] n=45266 width=64 errors=0
  [node 5 A5 d=1] n=45250 width=256 errors=0
  [node 6 A6 d=1] n=45289 width=256 errors=0
  [node 7 A7 d=1] n=45039 width=64 errors=0
  [node 8 A8 d=1] n=45001 width=16 errors=0
  [node 9 A9 d=1] n=45005 width=16 errors=0
{'depth': 0, 'train_acc': 0.98012, 'test_acc': 0.7054, 'params': 133916}
{'depth': 1, 'train_acc': 1.0, 'test_acc': 0.7043, 'params': 553349}
DONE  train 1.0000  test 0.7043  nodes 10  params 553349  root-only test 0.7054
device: cuda
train 50000  test 10000  classes 10  noisy labels 7592
  trunk epoch 1/3  loss 1.772 

In [ ]:
!python scripts/run_tree.py --dataset catsdogs --img_size 64 --trunk scratch_cnn --trunk_epochs 3 --out results/catsdogs_clean
!python scripts/run_tree.py --dataset catsdogs --img_size 64 --trunk scratch_cnn --trunk_epochs 3 --noise 0.15 --out results/catsdogs_noise15
!cp -r results {SAVE}/

device: cuda
downloading Cats vs Dogs ...
/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
train 19998  test 5000  classes 2  noisy labels 0
  trunk epoch 1/3  loss 0.651  train acc 0.623
  trunk epoch 2/3  loss 0.576  train acc 0.696
  trunk epoch 3/3  loss 0.527  train acc 0.738
features (19998, 512)  (37s)
[node 0 root d=0] n=19998 width=256 errors=2155
  [node 1 A0 d=1] n=10945 width=256 errors=446
  [node 13 A1 d=1] n=11208 width=256 errors=324
{'depth': 0, 'train_acc': 0.8922392239223922, 'test_acc': 0.758, 'params': 131846}
{'depth': 1, 'train_acc': 0.8922392239223922, 'test_acc': 0.758, 'params': 395020}
{'depth': 2, 'train_acc': 0.94999499949995, 'test_acc': 0.7376, 'params': 559504}
{'depth': 3, 'train_acc': 0.94999499949995, 'test_acc': 0.7376, 'params': 560019}
{'depth': 4, 'train_acc': 0.94999499949995, 'test_acc': 0.7376, 'params': 560536}
{'depth': 5, 'train_acc': 0.94999499949995, 'test_acc': 

## 5. Size minimisation and receptive fields

In [ ]:
for t in ['cifar10_clean','cifar10_noise15','catsdogs_clean','catsdogs_noise15']:
    !python scripts/minimize.py --run results/{t}
for t,d in [('cifar10_clean','cifar10'),('cifar10_noise15','cifar10'),('catsdogs_clean','catsdogs'),('catsdogs_noise15','catsdogs')]:
    !python scripts/visualize_rf.py --run results/{t} --dataset {d}
!cp -r results {SAVE}/

full tree {'params': 553349, 'nodes': 10, 'train_acc': 0.99994, 'test_acc': 0.7043}
{'min_pos': 1, 'params': 553349, 'nodes': 10, 'train_acc': 0.99994, 'test_acc': 0.7043}
{'min_pos': 2, 'params': 545122, 'nodes': 9, 'train_acc': 0.99992, 'test_acc': 0.7043}
{'min_pos': 3, 'params': 545122, 'nodes': 9, 'train_acc': 0.99992, 'test_acc': 0.7043}
{'min_pos': 5, 'params': 536895, 'nodes': 8, 'train_acc': 0.99984, 'test_acc': 0.7043}
{'min_pos': 10, 'params': 528668, 'nodes': 7, 'train_acc': 0.99974, 'test_acc': 0.7043}
{'min_pos': 20, 'params': 528668, 'nodes': 7, 'train_acc': 0.99974, 'test_acc': 0.7043}
{'min_pos': 50, 'params': 462870, 'nodes': 5, 'train_acc': 0.99822, 'test_acc': 0.7048}
{'min_pos': 100, 'params': 462870, 'nodes': 5, 'train_acc': 0.99822, 'test_acc': 0.7048}
{'min_pos': 200, 'params': 429971, 'nodes': 4, 'train_acc': 0.99616, 'test_acc': 0.7052}
{'min_pos': 500, 'params': 133898, 'nodes': 1, 'train_acc': 0.98008, 'test_acc': 0.7054}
{'sparsity': 0.0, 'params': 553349, 

## 6. Double descent (longest part)
Model-wise / epoch-wise with width-k ResNet-18 and label noise, then the tree-wise sweep.
`double_descent.py` and `tree_sweep.py` resume from their json if the session is interrupted (re-run the cell).
Use `--widths 2,4,8,16,32` and `--epochs 60` for a faster first look.

In [ ]:
!cp -r /content/tree_nn_project {SAVE}/code
!cp -r data {SAVE}/data
!mkdir -p {SAVE}/results && cp -r results/. {SAVE}/results/

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/tree_nn_results'
!cp -r {SAVE}/code /content/tree_nn_project
%cd /content/tree_nn_project
!cp -r {SAVE}/data data
!mkdir -p results && cp -r {SAVE}/results/. results/
!pip -q install reportlab scikit-learn matplotlib pillow

Mounted at /content/drive
/content/tree_nn_project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 82.2 MB/s eta 0:00:00


In [ ]:
!python scripts/double_descent.py --dataset cifar10 --noise 0.15 --n_train 15000 --out {SAVE}/results/dd_cifar10

  k= 16 ep    1  train err 0.6652  test err 0.6585
  k= 16 ep    5  train err 0.3529  test err 0.5822
  k= 16 ep   10  train err 0.0185  test err 0.6161
  k= 16 ep   15  train err 0.0002  test err 0.6181
  k= 16 ep   20  train err 0.0004  test err 0.6299
  k= 16 ep   25  train err 0.0047  test err 0.6321
  k= 16 ep   30  train err 0.0003  test err 0.6268
  k= 16 ep   35  train err 0.0000  test err 0.6171
  k= 16 ep   40  train err 0.0009  test err 0.6246
  k= 16 ep   45  train err 0.0723  test err 0.6647
  k= 16 ep   50  train err 0.0000  test err 0.6055
  k= 16 ep   55  train err 0.0014  test err 0.6159
  k= 16 ep   60  train err 0.0000  test err 0.6152
  k= 16 ep   65  train err 0.0003  test err 0.6123
  k= 16 ep   70  train err 0.0006  test err 0.6192
  k= 16 ep   75  train err 0.0001  test err 0.6196
  k= 16 ep   80  train err 0.0019  test err 0.6229
  k= 16 ep   85  train err 0.0004  test err 0.6118
  k= 16 ep   90  train err 0.0005  test err 0.6101
  k= 16 ep   95  train err 0.00

In [ ]:
!python scripts/double_descent.py --dataset cifar10 --noise 0.15 --n_train 15000 --out results/dd_cifar10
!cp -r results {SAVE}/

  k=  1 ep    1  train err 0.8773  test err 0.8725
  k=  1 ep    5  train err 0.8408  test err 0.8309
  k=  1 ep   10  train err 0.7939  test err 0.7819
  k=  1 ep   15  train err 0.7768  test err 0.7641
  k=  1 ep   20  train err 0.7675  test err 0.7535
  k=  1 ep   25  train err 0.7608  test err 0.7426
  k=  1 ep   30  train err 0.7486  test err 0.7343
  k=  1 ep   35  train err 0.7371  test err 0.7234
  k=  1 ep   40  train err 0.7289  test err 0.7160
  k=  1 ep   45  train err 0.7235  test err 0.7158
  k=  1 ep   50  train err 0.7200  test err 0.7119
  k=  1 ep   55  train err 0.7159  test err 0.7066
  k=  1 ep   60  train err 0.7121  test err 0.7048
  k=  1 ep   65  train err 0.7098  test err 0.7005
  k=  1 ep   70  train err 0.7060  test err 0.6993
  k=  1 ep   75  train err 0.7021  test err 0.7022
  k=  1 ep   80  train err 0.6999  test err 0.6974
  k=  1 ep   85  train err 0.6968  test err 0.6946
  k=  1 ep   90  train err 0.6939  test err 0.6953
  k=  1 ep   95  train err 0.69

In [ ]:
!python scripts/tree_sweep.py --dataset cifar10 --noise 0.15 --n_train 8000 \
    --widths 0,8,32,128,512 --max_depth 8 --max_nodes 1500 --steps 800 \
    --out {SAVE}/results/sweep_cifar10

  trunk epoch 1/3  loss 2.019  train acc 0.262
  trunk epoch 2/3  loss 1.824  train acc 0.363
  trunk epoch 3/3  loss 1.714  train acc 0.417
{'width': 0, 'params': 229670, 'n_nodes': 437, 'depth': 7, 'train_acc': 1.0, 'test_acc': 0.4212, 'root_test_acc': 0.5417, 'seconds': 346.6805639266968}
{'width': 8, 'params': 189369, 'n_nodes': 46, 'depth': 5, 'train_acc': 1.0, 'test_acc': 0.4133, 'root_test_acc': 0.5328, 'seconds': 43.50128102302551}
{'width': 32, 'params': 296413, 'n_nodes': 18, 'depth': 2, 'train_acc': 1.0, 'test_acc': 0.4261, 'root_test_acc': 0.4747, 'seconds': 16.192646980285645}
{'width': 128, 'params': 724904, 'n_nodes': 11, 'depth': 1, 'train_acc': 1.0, 'test_acc': 0.4445, 'root_test_acc': 0.4492, 'seconds': 8.941978693008423}
{'width': 512, 'params': 267786, 'n_nodes': 1, 'depth': 0, 'train_acc': 1.0, 'test_acc': 0.47, 'root_test_acc': 0.47, 'seconds': 2.520806312561035}
DONE


## 7. Report
1. Look at the figures/tables in `results/`, then write `interpretation.md` (start from `interpretation_template.md`).
2. Push the code to a PRIVATE GitHub repo and put its URL below.

In [ ]:
REPO_URL = 'https://github.com/ronakraichandani/MLOAssignment_2026EEY7568_Ronak'   # <- edit
!python make_report.py --results results --repo_url {REPO_URL} --interpretation interpretation.md --out report.pdf
from google.colab import files; files.download('report.pdf')
!cp report.pdf {SAVE}/

wrote report.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!ls {SAVE}/results
!ls {SAVE}/results/dd_cifar10 {SAVE}/results/sweep_cifar10

catsdogs_clean	  cifar10_clean    dd_cifar10
catsdogs_noise15  cifar10_noise15  sweep_cifar10
/content/drive/MyDrive/tree_nn_results/results/dd_cifar10:
dd_results.json  double_descent.png

/content/drive/MyDrive/tree_nn_results/results/sweep_cifar10:
sweep_results.json  tree_sweep.png


In [ ]:
!ls {SAVE}/results/catsdogs_clean {SAVE}/results/catsdogs_noise15

/content/drive/MyDrive/tree_nn_results/results/catsdogs_clean:
accuracy_vs_depth.png  minimize.json  results.json  tree.pt
feats.npz	       minimize.png   rf	    trunk.pt

/content/drive/MyDrive/tree_nn_results/results/catsdogs_noise15:
accuracy_vs_depth.png  minimize.json  results.json  tree.pt
feats.npz	       minimize.png   rf	    trunk.pt


In [ ]:
!cd {SAVE} && zip -qr results_light.zip results -x "*.pt" "*.npz" && ls -lh results_light.zip

-rw------- 1 root root 3.9M Oct  9 09:27 results_light.zip
